# Anomaly Detection Agent - Virtue Foundation Ghana

**Purpose:** Detect data quality anomalies and inconsistencies in facility records

**Anomaly Types:**
1. **High capacity, no doctors**: Facilities with >50 beds but 0 doctors
2. **Hospital without procedures**: Hospitals with no procedures listed
3. **Clinic with high capacity**: Clinics with >100 beds (unusual)
4. **Zero everything**: Facilities with 0 doctors, 0 beds, 0 specialties
5. **No contact info**: Facilities with no phone, email, or website
6. **Procedure-equipment mismatch**: Has procedures but no equipment
7. **Old facility, no data**: Established before 2000 but no specialties/procedures
8. **High doctor count outlier**: >50 doctors (may be data entry error)

**Input:** `virtue_foundation.ghana.facilities_silver`

**Output:** `virtue_foundation.ghana.facilities_anomalies`

## 1. Setup and Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import sys
sys.path.append('/Workspace/Users/anuragrc27@gmail.com/Databricks-AI-Agent')
from mlflow_utils.mlflow_utils import start_agent_run, log_anomaly_summary
import mlflow

# Configuration
CATALOG = "virtue_foundation"
SCHEMA = "ghana"
SOURCE_TABLE = "facilities_silver"
TARGET_TABLE = "facilities_anomalies"

print(f"Source: {CATALOG}.{SCHEMA}.{SOURCE_TABLE}")
print(f"Target: {CATALOG}.{SCHEMA}.{TARGET_TABLE}")

## 2. Load Silver Data

In [0]:
# Read Silver table
df_silver = spark.table(f"{CATALOG}.{SCHEMA}.{SOURCE_TABLE}")

total_facilities = df_silver.count()
print(f"✅ Loaded {total_facilities:,} facilities")

# Show sample
print("\nSample data:")
display(df_silver.select(
    "unique_id", "name", "facilityTypeId", 
    "numberDoctors", "capacity", "specialty_count",
    "has_procedures", "has_equipment", "has_any_contact"
).limit(5))

## 3. Apply Anomaly Detection Rules

Implement 8 data quality anomaly rules.

In [0]:
# Anomaly detection with optional MLflow tracking
try:
    # Try MLflow tracking if available
    mlflow_run = start_agent_run(
        "anomaly_detection",
        {
            "total_facilities": total_facilities,
            "anomaly_rules": 8,
            "source_table": f"{CATALOG}.{SCHEMA}.{SOURCE_TABLE}"
        }
    )
    use_mlflow = True
    print("✅ MLflow tracking enabled")
except Exception as e:
    print(f"⚠️  MLflow not available: {e}")
    print("   Continuing without experiment tracking...\n")
    use_mlflow = False
    mlflow_run = None

try:
    print("Applying anomaly detection rules...\n")
    
    df = df_silver
    
    # ========================================
    # ANOMALY 1: High capacity, no doctors
    # ========================================
    df = df.withColumn(
        "anomaly_high_capacity_no_doctors",
        F.when(
            (F.col("capacity") > 50) & 
            ((F.col("numberDoctors") == 0) | F.col("numberDoctors").isNull()),
            True
        ).otherwise(False)
    )
    
    # ========================================
    # ANOMALY 2: Hospital without procedures
    # ========================================
    df = df.withColumn(
        "anomaly_hospital_no_procedures",
        F.when(
            (F.col("facilityTypeId") == "hospital") &
            ((F.col("has_procedures") == False) | F.col("has_procedures").isNull()),
            True
        ).otherwise(False)
    )
    
    # ========================================
    # ANOMALY 3: Clinic with very high capacity
    # ========================================
    df = df.withColumn(
        "anomaly_clinic_high_capacity",
        F.when(
            (F.col("facilityTypeId") == "clinic") &
            (F.col("capacity") > 100),
            True
        ).otherwise(False)
    )
    
    # ========================================
    # ANOMALY 4: Zero everything (no data)
    # ========================================
    df = df.withColumn(
        "anomaly_zero_everything",
        F.when(
            ((F.col("numberDoctors") == 0) | F.col("numberDoctors").isNull()) &
            ((F.col("capacity") == 0) | F.col("capacity").isNull()) &
            (F.col("specialty_count") == 0),
            True
        ).otherwise(False)
    )
    
    # ========================================
    # ANOMALY 5: No contact information
    # ========================================
    df = df.withColumn(
        "anomaly_no_contact_info",
        F.when(
            (F.col("has_any_contact") == False) | F.col("has_any_contact").isNull(),
            True
        ).otherwise(False)
    )
    
    # ========================================
    # ANOMALY 6: Procedure-equipment mismatch
    # ========================================
    df = df.withColumn(
        "anomaly_procedure_no_equipment",
        F.when(
            (F.col("has_procedures") == True) &
            ((F.col("has_equipment") == False) | F.col("has_equipment").isNull()),
            True
        ).otherwise(False)
    )
    
    # ========================================
    # ANOMALY 7: Old facility, no data
    # ========================================
    df = df.withColumn(
        "anomaly_old_facility_incomplete",
        F.when(
            (F.col("yearEstablished") < 2000) &
            (F.col("specialty_count") == 0) &
            ((F.col("has_procedures") == False) | F.col("has_procedures").isNull()),
            True
        ).otherwise(False)
    )
    
    # ========================================
    # ANOMALY 8: Unusually high doctor count
    # ========================================
    df = df.withColumn(
        "anomaly_high_doctor_count",
        F.when(
            F.col("numberDoctors") > 50,
            True
        ).otherwise(False)
    )
    
    # ========================================
    # COUNT TOTAL ANOMALIES PER FACILITY
    # ========================================
    anomaly_cols = [
        "anomaly_high_capacity_no_doctors",
        "anomaly_hospital_no_procedures",
        "anomaly_clinic_high_capacity",
        "anomaly_zero_everything",
        "anomaly_no_contact_info",
        "anomaly_procedure_no_equipment",
        "anomaly_old_facility_incomplete",
        "anomaly_high_doctor_count"
    ]
    
    # Sum of all anomaly flags
    df = df.withColumn(
        "total_anomaly_flags",
        sum([F.when(F.col(c) == True, 1).otherwise(0) for c in anomaly_cols])
    )
    
    # Add detection timestamp
    df = df.withColumn(
        "anomaly_detected_at",
        F.current_timestamp()
    )
    
    print("✅ Anomaly detection complete\n")
    
    # Calculate anomaly counts
    anomaly_counts = {}
    for col in anomaly_cols:
        count = df.filter(F.col(col) == True).count()
        anomaly_type = col.replace("anomaly_", "")
        anomaly_counts[anomaly_type] = count
        print(f"  {anomaly_type}: {count:,} facilities")
    
    # Log to MLflow if available
    if use_mlflow and mlflow_run:
        try:
            log_anomaly_summary(anomaly_counts)
        except Exception as e:
            print(f"\n⚠️  MLflow logging failed: {e}")
    
    # Overall stats
    facilities_with_anomalies = df.filter(F.col("total_anomaly_flags") > 0).count()
    anomaly_rate = facilities_with_anomalies / total_facilities * 100
    
    print(f"\n🚨 Total facilities with anomalies: {facilities_with_anomalies:,} ({anomaly_rate:.1f}%)")
    print(f"   Clean facilities: {total_facilities - facilities_with_anomalies:,}")
    
finally:
    # Close MLflow run if it was opened
    if use_mlflow and mlflow_run:
        try:
            mlflow.end_run()
        except:
            pass

## 4. Save Anomaly Detection Results

In [0]:
# Write full table with anomaly flags
full_table_name = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(full_table_name)

print(f"✅ Anomalies saved to: {full_table_name}")
print(f"   Records: {df.count():,}")

In [0]:
%sql
COMMENT ON TABLE virtue_foundation.ghana.facilities_anomalies IS
'Facilities with data quality anomaly flags.
Includes 8 anomaly types: capacity/doctor mismatch, missing procedures, contact info, etc.';

## 5. Analyze Anomaly Patterns

In [0]:
df_anomalies = spark.table(full_table_name)

print("="*80)
print("ANOMALY ANALYSIS")
print("="*80)

# Distribution by anomaly count
print("\nFacilities by Anomaly Count:")
df_anomalies.groupBy("total_anomaly_flags") \
    .count() \
    .orderBy("total_anomaly_flags") \
    .show()

# Most common anomaly combinations
print("\nMost Common Anomaly Types:")
for col in anomaly_cols:
    count = df_anomalies.filter(F.col(col) == True).count()
    pct = count / total_facilities * 100
    anomaly_name = col.replace("anomaly_", "").replace("_", " ").title()
    print(f"  {anomaly_name}: {count:,} ({pct:.1f}%)")

print("\n" + "="*80)

In [0]:
# Anomaly rate by facility type
print("\nAnomaly Rate by Facility Type:\n")

df_anomalies.groupBy("facilityTypeId") \
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("total_anomaly_flags") > 0, 1).otherwise(0)).alias("with_anomalies"),
        (F.sum(F.when(F.col("total_anomaly_flags") > 0, 1).otherwise(0)) * 100.0 / F.count("*")).alias("anomaly_rate")
    ) \
    .orderBy(F.col("anomaly_rate").desc()) \
    .show()

In [0]:
# Show facilities with multiple anomalies
print("\nFACILITIES WITH MULTIPLE ANOMALIES:\n")

high_anomaly = df_anomalies.filter(F.col("total_anomaly_flags") >= 3) \
    .orderBy(F.col("total_anomaly_flags").desc()) \
    .limit(10) \
    .collect()

for i, row in enumerate(high_anomaly, 1):
    print(f"{i}. {row.name} ({row.facilityTypeId})")
    print(f"   Anomalies: {row.total_anomaly_flags}")
    
    # List specific anomalies
    anomalies_found = []
    for col in anomaly_cols:
        if row[col]:
            anomalies_found.append(col.replace("anomaly_", "").replace("_", " "))
    print(f"   Issues: {', '.join(anomalies_found)}")
    print()

In [0]:
# Anomaly patterns by region
print("\nANOMALY RATE BY REGION:\n")

df_anomalies.groupBy("address_stateOrRegion") \
    .agg(
        F.count("*").alias("total_facilities"),
        F.avg("total_anomaly_flags").alias("avg_anomalies_per_facility"),
        (F.sum(F.when(F.col("total_anomaly_flags") > 0, 1).otherwise(0)) * 100.0 / F.count("*")).alias("anomaly_rate")
    ) \
    .orderBy(F.col("anomaly_rate").desc()) \
    .show(10)

## 6. Export Priority Cases for Manual Review

Identify facilities that need urgent data quality attention.

In [0]:
# High-priority anomalies needing manual review
priority_cases = df_anomalies.filter(
    (F.col("total_anomaly_flags") >= 2) |
    (F.col("anomaly_hospital_no_procedures") == True) |
    (F.col("anomaly_high_capacity_no_doctors") == True)
)

priority_count = priority_cases.count()

print(f"🚨 Priority cases for manual review: {priority_count:,}")
print(f"\nCriteria:")
print(f"  • 2+ anomalies detected")
print(f"  • Hospital without procedures listed")
print(f"  • High bed capacity but no doctors")

# Show sample
print("\nSample priority cases:")
display(priority_cases.select(
    "unique_id", "name", "facilityTypeId", "address_stateOrRegion",
    "total_anomaly_flags",
    "anomaly_hospital_no_procedures",
    "anomaly_high_capacity_no_doctors",
    "anomaly_no_contact_info"
).limit(10))

## ✅ Anomaly Detection Complete!

**What was detected:**
* ✅ 8 anomaly types checked
* ✅ {facilities_with_anomalies} facilities flagged
* ✅ {priority_count} priority cases identified
* ✅ Results tracked with MLflow

**Common Anomalies:**
1. **No contact info**: Most common (~40-60% of facilities)
2. **Zero everything**: Facilities with no data at all
3. **Hospital without procedures**: Critical data gap
4. **High capacity, no doctors**: Resource planning issue

**Next Steps:**
1. Review priority cases manually
2. Contact facilities to update missing information
3. Use for data quality improvement campaigns
4. Schedule regular anomaly detection runs